# Day 6: Decoding Generative UI Agents (v0, Bolt.new, Lovable)

## Core Problem

Traditional code assistants generate code, but they cannot verify whether it actually works or matches the user's intent.

Generative UI agents solve this by converting a natural language prompt into a running application, rendering it, observing failures, and iteratively repairing it.

Typical workflow:

```text
Prompt
    ↓
Generate Code
    ↓
Run Application
    ↓
Render UI
    ↓
Observe Errors / UI
    ↓
Repair
    ↓
Repeat
```

---

# Why an Agent is Needed

A chatbot only generates code.

A generative UI agent performs an entire workflow:

- Understand user intent
- Plan project structure
- Generate multiple files
- Install dependencies
- Execute the application
- Detect compiler/runtime errors
- Render a live preview
- Modify code based on feedback
- Repeat until acceptable

This requires planning, execution, observation, and iteration, making it an agent rather than a simple text generator.

---

# Architecture

```text
User Prompt
      ↓
Intent Parsing
      ↓
Project Planning
      ↓
Multi-file Code Generation
      ↓
Sandboxed Execution
      ↓
Build & Run
      ↓
Render Preview
      ↓
Collect Errors
      ↓
LLM Repairs Code
      ↓
Repeat
```

### Intent Parsing

Extracts:

- Framework
- UI requirements
- Backend requirements
- Database requirements
- Authentication
- APIs
- User expectations

---

### Project Planning

Creates a file-level execution plan before writing code.

Example:

```text
/app
/components
/lib
/package.json
```

---

### Code Generation

Produces:

- Frontend
- Backend
- Configuration
- Dependency files
- Styles
- Routing

---

### Sandboxed Execution

Generated code is never executed directly on the user's machine.

Instead it runs inside an isolated environment.

Benefits:

- Prevents malicious code execution
- Protects user files
- Prevents secret leakage
- Enables automatic build and testing
- Allows autonomous debugging

A sandbox is non-negotiable because an agent that writes code must execute it safely to verify correctness.

---

### Generate → Run → Repair Loop

```text
Generate
    ↓
Execute
    ↓
Observe Errors
    ↓
Repair
    ↓
Execute Again
```

The model does not assume the code is correct. It verifies it through execution.

---

# Verification Pipeline

A production agent does not trust generated code.

It verifies it through multiple layers.

## 1. Type Checking

Purpose:

Verify contracts between files.

Catches:

- Incorrect props
- Wrong function signatures
- Invalid types
- Missing exports

Example:

Changing:

```tsx
price
```

to

```tsx
amount
```

breaks every file still using `price`.

TypeScript detects this immediately.

---

## 2. Linting

Purpose:

Detect bad coding practices.

Examples:

- Unused variables
- Missing dependencies
- Dangerous patterns
- Style violations

Linting improves maintainability, not runtime correctness.

---

## 3. Unit Tests

Purpose:

Verify individual functions or components.

Checks:

- One function
- One class
- One React component

---

## 4. Integration Tests

Purpose:

Verify multiple modules working together.

Example:

```text
Login
    ↓
Authentication
    ↓
Database
    ↓
Dashboard
```

Each unit may work independently while the overall workflow fails.

---

## 5. Build

Purpose:

Verify the entire project compiles successfully.

Catches:

- Missing imports
- Missing packages
- Syntax errors
- Compilation failures

A successful build only guarantees that the project can compile.

It does **not** guarantee that the application runs correctly.

---

## 6. Smoke Test

Purpose:

Verify the application actually starts.

Checks:

- App launches
- Homepage loads
- No immediate crashes
- Main routes respond

Catches:

- Runtime exceptions
- Missing environment variables
- Infinite render loops
- API startup failures

A smoke test does **not** verify UI quality.

---

# Why Build Is Not Enough

Build only answers:

> Can the project compile?

Smoke test answers:

> Can the application actually run?

Examples that pass build but fail at runtime:

- Undefined objects
- Missing environment variables
- Backend unavailable
- Runtime exceptions
- Infinite React render loops

Compile-time correctness is not runtime correctness.

---

# UI Quality Validation

A smoke test cannot detect visual problems.

Example:

The prompt asks for:

> Small button on the right.

Generated output:

> Huge black card in the center.

The application runs correctly.

No compiler errors.

No runtime errors.

Smoke test passes.

The UI is still wrong.

Modern UI agents increasingly use:

```text
Generate
    ↓
Render
    ↓
Capture Screenshot
    ↓
Vision Model Evaluation
    ↓
Compare Against Prompt
    ↓
Repair UI
```

This validates appearance rather than execution.

---

# Failure Modes

## Hallucinated APIs

Generated functions or libraries do not exist.

Recovered using build failures and runtime errors.

---

## Ambiguous User Intent

The generated UI differs from user expectations.

Resolved using preview-feedback iterations.

---

## Multi-file Inconsistency

Editing one file breaks another untouched file.

Recovered using:

- Type checking
- Build
- Tests
- Dependency analysis

---

## Tool Failure

Examples:

- Package installation failure
- Dev server crash
- Network issues

Recovered through retries or alternate implementations.

---

## Runtime Failure

Examples:

- White screen
- API crash
- React exception

Recovered by observing runtime logs and repairing code.

---

# Why Untouched Files Can Break

Changing one component can modify:

- Props
- Function signatures
- Types
- Exports
- Shared utilities

Any importing file may fail despite never being edited.

Context retrieval predicts possible impact.

Execution and verification prove actual impact.

Good interview line:

> Context tells the agent what might break. Execution and testing prove what actually broke.

---

# Cursor vs v0 / Bolt

## Cursor

Primary focus:

Existing repositories.

Workflow:

```text
Understand Repo
      ↓
Plan Multi-file Changes
      ↓
Apply Diffs
      ↓
Run Tests
```

Strengths:

- Refactoring
- Backend development
- Existing codebases
- Multi-file engineering

---

## v0 / Bolt

Primary focus:

Generating and iterating complete applications.

Workflow:

```text
Prompt
      ↓
Generate App
      ↓
Render Preview
      ↓
Observe
      ↓
Repair
```

Strengths:

- Rapid prototyping
- UI generation
- Full-stack MVPs
- Live preview

Cursor is repo-first.

v0/Bolt are preview-first.

---

# Bolt vs Lovable

Architecturally they are nearly identical.

Shared pipeline:

```text
Prompt
      ↓
Generate Frontend
      ↓
Generate Backend
      ↓
Sandbox
      ↓
Build
      ↓
Preview
      ↓
Repair
```

Difference lies primarily in product positioning.

### Bolt

Optimized for:

- Full-stack engineering
- Software development
- Terminal workflows
- Package management
- Debugging

### Lovable

Optimized for:

- Product creation
- UI polish
- UX
- Rapid MVP development
- Low-code experience

The underlying agent architecture is largely the same.

---

# Key Interview Takeaways

- Generative UI agents are execution-driven, not text-driven.
- They verify code by running it instead of trusting model output.
- Sandboxing enables safe autonomous execution.
- Build success does not imply runtime correctness.
- Smoke tests validate execution, not visual correctness.
- Visual correctness increasingly requires render plus vision evaluation.
- Production agents rely on layered verification including type checks, linting, tests, builds, runtime execution, and UI inspection.
- Cursor optimizes software engineering on existing repositories.
- v0, Bolt, and Lovable optimize prompt-to-application generation with rapid preview-feedback loops.